# Hierarchical Multiple Testing

This is a runnable, toy-data example of the `qtl_association_testing` post-processing step. It performs hierarchical multiple-testing correction on the per-gene cis-QTL association results produced by TensorQTL, then archives the bulky intermediate files.

## Description

Given the per-gene association output from a cis-QTL scan, this pipeline implements a three-step hierarchical procedure: (1) local adjustment of the p-values of all cis-variants within each gene, (2) global adjustment of the minimum adjusted p-value across genes, and (3) identification of significant xQTL whose locally adjusted p-value falls below the threshold. It also reorganises the intermediate TensorQTL files into an archive folder for book-keeping or deletion.

The same `default` workflow handles three flavours of upstream results: the **genetic (cis) effect** (shown below), the **interaction effect** (run on `*.cis_qtl_top_assoc.txt.gz` with `--maf-cutoff 0 --cis-window 0` and interaction p/q-value patterns), and **quantile QTL**. This example demonstrates the genetic-effect flavour on toy data. (cf. Huang et al. 2018)

**Timing:** ~1-2 min on typical compute infrastructure.

## Input Files

| Role | Toy file | Produced by |
|------|----------|-------------|
| Per-gene cis association | `output/tensorqtl_cis/*.cis_qtl.regional.tsv.gz` | TensorQTL cis scan |
| Full variant-level pairs | `output/tensorqtl_cis/*.cis_qtl.pairs.tsv.gz` | TensorQTL cis scan |
| Gene coordinates | `input/reference_data/look_up_gene_id.tsv` (`chr`, `start`, `end`, `gene_id`) | reference |

## Output Files

Written under `--output-dir` (significance results) and `--archive-dir` (moved intermediates):

| File | Meaning |
|------|---------|
| `*_multiple_testing_consolidated.rds` | Consolidated multiple-testing results object |
| `*.cis_regional.summary.txt` | Per-gene regional summary after correction |
| `*significant_qtl*.tsv.gz` | Significant variant-level QTL (Bonferroni / BH / permutation) |
| `*significant_events*.tsv.gz` | Significant gene-level events |
| `archive/.../*.parquet` | Bulky TensorQTL intermediates moved out of the working dir |

On the 6-sample toy data the correction runs cleanly and produces the consolidated object and summary; no events pass the genome-wide significance threshold, which is expected at this tiny scale.

## Steps

Run the hierarchical correction on the toy cis results. The MAF / cis-window options are non-zero, so the program recomputes the variant counts from the full pairs file before applying the Bonferroni correction. On this toy data the distance columns in the TensorQTL output are named `tss_distance` and `tes_distance`, so we point `--tss-dist-col` / `--tes-dist-col` at them.

In [ ]:
sos run pipeline/qtl_association_postprocessing.ipynb default \
    --cwd output/tensorqtl_cis \
    --gene-coordinates input/reference_data/look_up_gene_id.tsv \
    --sub-dir . \
    --tss-dist-col tss_distance --tes-dist-col tes_distance \
    --maf-cutoff 0.01 --cis-window 1000000 \
    --regional-pattern "*.cis_qtl.regional.tsv.gz$" \
    --output-dir output/hierarchical_multi_test/output \
    --archive-dir output/hierarchical_multi_test/archive --enable-archive True \
    --pecotmr-path ../pecotmr \
    -s force

## Command interface

Show all workflow options and their defaults:

In [ ]:
sos run pipeline/qtl_association_postprocessing.ipynb -h

## Workflow implementation

The cells below are the unmodified SoS workflow definition that the example above calls.

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
[global]
parameter: cwd = path(".")
# code/script dir that holds the pecotmr_integration wrappers
parameter: modular_script_dir = path
parameter: output_dir = path
parameter: sub_dir = path(".")
# labels for the assembled QtlSumStats
parameter: study = "study"
parameter: context = "context"
parameter: genome = "hg38"
parameter: maf_cutoff = 0.01
parameter: cis_window = 1000000
parameter: af_col = "af"
parameter: pvalue_col = "pvalue"
parameter: molecular_id_col = "molecular_trait_object_id"
# drop pairs with p above this before assembling (keeps the object small; the
# per-gene min variant is always retained so the Bonferroni min is exact)
parameter: pvalue_cutoff = 0.05
parameter: fdr_threshold = 0.05
# list.files() regex patterns to locate the inputs in the work dir
parameter: regional_pattern = "cis_qtl[.]regional[.]tsv[.]gz$"
parameter: qtl_pattern = "cis_qtl[.]pairs[.]tsv[.]gz$"
parameter: n_variants_suffix = "cis_n_variants_stats[.]tsv[.]gz$"
parameter: enable_archive = False

work_dir = cwd if str(sub_dir) == "." else path(f"{cwd:a}/{sub_dir}")


In [ ]:
[default]
# Hierarchical multiple-testing correction of the cis-QTL association results,
# via pecotmr::qtlAssociationPostprocess (the wrapper reads the per-gene regional
# + per-variant pairs, assembles a per-gene QtlSumStats, and writes the enriched
# consolidated RDS + per-method regional / significant-QTL / event / summary
# tables). Replaces the old source(pecotmr/inst/code/tensorqtl_postprocessor.R).
output: f"{output_dir:a}/{cwd:b}.qtl_association_postprocessing.rds"
bash: expand = "${ }"
    Rscript ${modular_script_dir:a}/pecotmr_integration/qtl_association_postprocessing.R \
        --cwd ${work_dir:a} \
        --regional-pattern "${regional_pattern}" \
        --qtl-pattern "${qtl_pattern}" \
        --n-variants-suffix "${n_variants_suffix}" \
        --maf-cutoff ${maf_cutoff} --cis-window ${cis_window} \
        --fdr-threshold ${fdr_threshold} --pvalue-cutoff ${pvalue_cutoff} \
        --pvalue-col "${pvalue_col}" --af-col "${af_col}" \
        --study "${study}" --context "${context}" --genome "${genome}" \
        --output ${_output:a} --output-dir ${output_dir:a}
